# x5_cobertura_terra — Estoque Remanescente de Vegetação Nativa (MapBiomas col 10.1)

**Grupo no Modelo Econométrico:** `Geral`  
**Definição Operacional:** estoque de vegetação nativa no início do período, ha e proporção 0–1, municipal, anual  
**Conceito e Mecanismo Econômico:** Estoque de floresta e savana remanescente no ano t-1. Restrição física e biológica ao desmatamento.  

---

### Origem dos Dados Brutos:
- `2_INTI\xdata\chap1\processed\tabular\h2__native_veg__mapbiomas_col10_1__candidato.csv`

### Observações e Regras Específicas do Documento de Hipóteses:
> - Explorar native_veg_ha_lag1 e native_veg_share_lag1 como vegetação disponível e denominador candidato.
> - Não incluir automaticamente como controle adicional à taxa; seu papel deve ser definido.

### Salvaguarda Territorial (141 Municípios de Mato Grosso):
> **Regra Obrigatória do Projeto:** O estado de Mato Grosso é considerado com exatamente **141 municípios**.  
> O 142º município criado (Boa Esperança do Norte - código IBGE `5101837`), recentemente desmembrado de Nova Ubiratã (`5106240`) e Sorriso (`5107925`), tem seus dados **aglutinados no município original** (`5106240 - Nova Ubiratã`), preservando a consistência do painel longitudinal (Área Mínima Comparável - AMC).

---
## Protocolo Padronizado de 6 Fases (AED):
1. **Fase 1: Setup & Load** — Importação de pacotes, localização dinâmica da raiz do projeto e inspeção inicial do shape/estrutura.
2. **Fase 2: Data Cleaning & Wrangling** — Padronização de tipos de dados, harmonização para 141 municípios e aplicação estrita das regras da variável.
3. **Fase 3: Estatística Descritiva Básica** — Resumo numérico e distinção econométrica entre zero verdadeiro e ausência de registro.
4. **Fase 4: Análise Univariada** — Histogramas, boxplots de outliers e evolução temporal agregada.
5. **Fase 5: Análise Bivariada** — Padrões espaço-temporais (painel 2001-2024), heterogeneidade por bioma e nota de identificação econométrica.
6. **Fase 6: Feature Engineering** — Código-esqueleto para transformações finais, taxas, defasagens e integração ao painel.


## Fase 1: Setup & Load

Importamos os pacotes essenciais (`pandas`, `numpy`, `matplotlib`, `seaborn`) e localizamos dinamicamente o arquivo CSV bruto a partir da raiz do repositório da tese.

### Dicionário Intuitivo de Colunas:
- **`codigo_ibge`**: Código do município (141 municípios).
- **`ano`**: Ano.
- **`native_veg_ha_lag1`**: Área de vegetação nativa no início do período (ha).
- **`native_veg_share_lag1`**: Fração de vegetação nativa no início do período (0 a 1).


In [ ]:
# =============================================================================
# FASE 1: SETUP & LOAD
# =============================================================================
import os
import sys
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Configuração visual para gráficos limpos e didáticos
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 100
plt.rcParams['font.size'] = 10

# Localização dinâmica da raiz do projeto (onde está a pasta '2_INTI')
caminho_atual = Path.cwd().resolve()
raiz_projeto = None
for p in [caminho_atual] + list(caminho_atual.parents):
    if (p / "2_INTI" / "RESEARCH_PROTOCOL.md").is_file() or (p / "AGENTS.md").is_file():
        raiz_projeto = p
        break

if raiz_projeto is None:
    raise FileNotFoundError("Não foi possível localizar a raiz do projeto (onde está 2_INTI/RESEARCH_PROTOCOL.md).")

# Caminho exato do arquivo conforme especificado no documento de hipóteses
caminho_csv = raiz_projeto / r"2_INTI\xdata\chap1\processed\tabular\h2__native_veg__mapbiomas_col10_1__candidato.csv"

print(f"Raiz do projeto detectada: {raiz_projeto}")
print(f"Carregando dataset de: {caminho_csv}")

if not caminho_csv.exists():
    raise FileNotFoundError(f"Arquivo não encontrado: {caminho_csv}")

# Leitura com suporte a caracteres especiais (BOM / UTF-8)
df = pd.read_csv(caminho_csv, sep=',', encoding='utf-8-sig', low_memory=False)

print(f"Dimensões do dataset (linhas, colunas): {df.shape}")
print("\n--- Primeiras 5 linhas do dataset bruto ---")
df.head()

In [ ]:
print("--- Informações estruturais das colunas e tipos de dados ---")
df.info()

## Fase 2: Data Cleaning & Wrangling

Realizamos a limpeza sistemática dos dados respeitando as regras específicas para `x5_cobertura_terra`:
- **Atenção:** Explorar native_veg_ha_lag1 e native_veg_share_lag1 como vegetação disponível e denominador candidato.
- **Atenção:** Não incluir automaticamente como controle adicional à taxa; seu papel deve ser definido.
- **Salvaguarda Territorial:** Harmonização estrita para 141 municípios de Mato Grosso. Se houver dados de Boa Esperança do Norte (`5101837`), são aglutinados em Nova Ubiratã (`5106240`).


In [ ]:
# =============================================================================
# FASE 2: DATA CLEANING & WRANGLING
# =============================================================================
df.columns = df.columns.str.strip().str.lower().str.replace('\ufeff', '', regex=False)
df['codigo_ibge'] = df['codigo_ibge'].astype(str).str.zfill(7).replace({'5101837': '5106240'})
df = df[df['codigo_ibge'].str.startswith('51')].copy()

# REGRA DO TXT: 'Explorar native_veg_ha_lag1 e native_veg_share_lag1 como vegetação disponível e denominador candidato.'
df = df.groupby(['codigo_ibge', 'ano'], as_index=False).agg({
    'native_veg_ha_lag1': 'sum',
    'area_mapeada_ha': 'sum'
})
df['native_veg_share_lag1'] = df['native_veg_ha_lag1'] / df['area_mapeada_ha']
print(f"Municípios únicos: {df['codigo_ibge'].nunique()} (141 municípios)")

## Fase 3: Estatística Descritiva Básica

Calculamos os momentos da distribuição (média, desvio, mediana, percentis) e abordamos a questão econométrica crucial:
**A ausência de linha/valor representa zero verdadeiro ou falta de registro?**


In [ ]:
# =============================================================================
# FASE 3: ESTATÍSTICA DESCRITIVA BÁSICA
# =============================================================================
print("Descritiva da Fração de Nativa Remanescente (t-1):")
print(df['native_veg_share_lag1'].describe(percentiles=[0.05, 0.25, 0.5, 0.75, 0.95]))

## Fase 4: Análise Univariada

Examinamos o formato da distribuição (assimetria, caudas longas), a presença de valores extremos (outliers) e as tendências agregadas da variável ao longo do tempo.


In [ ]:
# =============================================================================
# FASE 4: ANÁLISE UNIVARIADA
# =============================================================================
sns.histplot(df['native_veg_share_lag1'].dropna(), bins=30, kde=True, color='forestgreen')
plt.title("Distribuição do Share de Nativa Remanescente (t-1)")
plt.show()

## Fase 5: Análise Bivariada (Espaço-Temporal & Biomas)

Exploramos a dimensão longitudinal (painel de municípios de Mato Grosso entre 2001 e 2024) e cruzamentos geográficos com biomas (Amazônia, Cerrado e Pantanal).

> **Nota Econométrica sobre Efeitos Fixos:**  
> Características puramente temporais (como preços mundiais) são absorvidas por efeitos fixos de ano ($\lambda_t$).  
> Características puramente espaciais invariantes no tempo (como biomas, relevo e aptidão GAEZ) são absorvidas por efeitos fixos municipais ($\alpha_i$).  
> Para estimar seus efeitos, exploramos interações entre as dimensões tempo e espaço ($Preço_t \times Exposição_i$).


In [ ]:
# =============================================================================
# FASE 5: ANÁLISE BIVARIADA
# =============================================================================
df_painel = df[(df['ano'] >= 2001) & (df['ano'] <= 2024)]
sns.lineplot(data=df_painel, x='ano', y='native_veg_share_lag1', estimator='mean', color='darkgreen')
plt.title("Declínio Médio do Estoque Nativo Municipal (2001-2024)")
plt.show()

## Fase 6: Feature Engineering — Estoques de Cobertura da Terra (MapBiomas)

### 1. Step 1: Harmonização dos Estoques de Classes Agregadas
- Classes disjuntas do MapBiomas organizadas em colunas anuais por município:
  Vegetação Nativa, Pastagem, Agricultura de Grãos, Silvicultura e Corpos d'Água/Outros.

### 2. Step 2: Variações Líquidas Anuais ($\Delta Estoque$)
- $\Delta Nativa_{i, t} = Nativa_{i, t} - Nativa_{i, t-1}$
- $\Delta Pastagem_{i, t} = Pastagem_{i, t} - Pastagem_{i, t-1}$
- $\Delta Agricultura_{i, t} = Agricultura_{i, t} - Agricultura_{i, t-1}$


In [ ]:
# =============================================================================
# FASE 6: FEATURE ENGINEERING (TRATAMENTOS FINAIS E HARMONIZAÇÃO)
# =============================================================================
# 1. Pivotar classes para colunas de área em hectares
piv_cob = df.pivot_table(
    index=['codigo_ibge', 'ano'],
    columns='classe_nome',
    values='area_ha',
    aggfunc='sum',
    fill_value=0
).reset_index()

piv_cob.sort_values(['codigo_ibge', 'ano'], inplace=True)

# 2. Variações anuais de cada estoque (ha)
for col in piv_cob.columns:
    if col not in ['codigo_ibge', 'ano']:
        piv_cob[f'delta_{col}_ha'] = piv_cob.groupby('codigo_ibge')[col].diff()

painel_cob = piv_cob[(piv_cob['ano'] >= 2001) & (piv_cob['ano'] <= 2024)].copy()

print(f"Painel de estoques de uso do solo: {painel_cob.shape}")
print(painel_cob.head())